<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B06%5D%20-%20Workshop%20Clustering/Workshop_Clustering_Spotify_RESUELTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎧 Workshop · Clustering end-to-end con Spotify - VERSIÓN RESUELTA

**Máster en Data Science · The Valley**

---

## El encargo

Sois el equipo de Data Science de Spotify. El equipo de producto quiere lanzar **playlists automáticas por tipo de canción**, sin etiquetar nada a mano. Nos piden tres entregables:

1. **Una segmentación del catálogo**: grupos de canciones que se parecen entre sí, con método y número de grupos justificados.
2. **Una ficha de segmentos**: qué es cada grupo, un nombre que producto entienda y una acción (playlist, campaña...) por grupo.
3. **Un mecanismo de asignación**: mañana entran canciones nuevas al catálogo y hay que colocarlas en su segmento sin reentrenar nada.

No hay target: nadie nos dice cuáles son los grupos "correctos". Esto es **aprendizaje no supervisado** de principio a fin.

> **Nota del profesor:** esta versión incluye el código resuelto y respuestas orientativas. Los resultados exactos (silhouettes, perfiles, nombres) pueden variar ligeramente si cambias la muestra, la semilla o el K.


## 🗺️ El plan de trabajo de ML no supervisado

Este plan sirve para **cualquier proyecto no supervisado** (segmentar clientes, detectar anomalías, agrupar productos...). Guárdatelo: hoy lo recorremos entero.

| # | Fase | Pregunta que respondes | Entregable |
|---|------|------------------------|------------|
| 1 | Problema de negocio | ¿Qué decisión va a tomar alguien con esto? | Objetivo + unidad de análisis (aquí: la canción) |
| 2 | EDA | ¿Qué datos tengo y en qué estado están? | Lista de problemas y decisiones sobre los datos |
| 3 | Ingeniería de variables | ¿Qué variables entran al modelo y en qué forma? | Matriz X limpia, transformada y **escalada** |
| 4 | Reducción de dimensión | ¿Puedo comprimir sin perder señal? ¿Cómo se ven mis datos? | Nº de componentes + visualización 2D (PCA) |
| 5 | Modelado | ¿Qué algoritmo y cuántos grupos? | Modelos + K justificado (codo, silhouette, dendrograma) |
| 6 | Comparación | ¿Qué solución es mejor **para este caso**? | Modelo elegido y por qué |
| 7 | Profiling | ¿Qué es cada grupo y qué hacemos con él? | Ficha de segmentos: nombre + acción |
| 8 | Activación | ¿Cómo lo usa el producto mañana? | Asignación de datos nuevos (aquí: KNN) |

Fíjate en el orden: **escalar antes de medir distancias**, **reducir antes de visualizar**, **perfilar antes de accionar**. Cambiar el orden es la fuente de errores más habitual.


## 🎮 Cómo funciona el workshop (léelo, son 60 segundos)

- **2,5 horas: setup + 6 fases + 1 bonus.** Cada fase tiene un tiempo orientativo y termina en un **CHECKPOINT**: una celda que verifica tu trabajo automáticamente y te dice si puedes cerrar la fase. Supera los 6 y habrás hecho un proyecto no supervisado completo.
- **Los checkpoints no dan la solución**: solo comprueban que tus variables existen y tienen sentido. Usa los nombres de variable que pide cada fase (`X_scaled`, `labels_km`...): son el contrato entre tu código y el checkpoint.
- **Pistas plegables 💡**: cada TODO tiene pistas (la 1 orienta el enfoque, la 2 da funciones concretas). Ábrelas solo si te atascas: primero inténtalo sin ellas.
- **[MÍNIMO] y [EXTRA]**: si vas justo de tiempo, completa solo lo marcado [MÍNIMO] y cierra la fase. Es mejor cerrar todas las fases que perfeccionar una sola.
- Si trabajáis en parejas: uno teclea y otro navega/piensa, y rotáis en cada fase.


---
# Fase 0 · Setup y datos (10 min)

Todo el código de esta fase viene dado: **ejecútalo y entiéndelo**, no hay que escribir nada.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.width", 140)


In [ ]:
# Sistema de checkpoints del workshop: no lo edites, solo ejecútalo.
CHECKPOINTS = {}

def checkpoint(n, checks):
    """checks = lista de (mensaje si falla, condición que debe cumplirse)"""
    fallos = [msg for msg, ok in checks if not ok]
    if fallos:
        CHECKPOINTS[n] = False
        print(f"❌ Checkpoint {n}: aún no. Revisa esto:")
        for msg in fallos:
            print("   -", msg)
    else:
        CHECKPOINTS[n] = True
        print(f"✅ CHECKPOINT {n} SUPERADO. Fase cerrada, puedes avanzar.")
    print(f"   Progreso: {sum(bool(v) for v in CHECKPOINTS.values())}/6")


In [ ]:
# Dataset de Spotify: 114.000 canciones con features de audio.
# Si la descarga remota falla, baja el zip a mano y cambia DATA_URL por la ruta local.
DATA_URL = "https://drive.google.com/uc?export=download&id=1a9pA0dFbwVH2i6UZjgwcicvcD_SCmG3I"

df_full = pd.read_csv(DATA_URL, compression="zip")
print(df_full.shape)
df_full.head(3)


In [ ]:
# Limpieza mínima de partida (fíjate en el porqué de cada línea):
# - 'Unnamed: 0' es un índice exportado por error -> fuera
# - la misma canción aparece repetida en varios géneros -> nos quedamos una por track_id
# - hay un puñado de filas con textos nulos -> fuera
df_full = (df_full
           .drop(columns=["Unnamed: 0"])
           .drop_duplicates(subset="track_id")
           .dropna()
           .reset_index(drop=True))
print(f"Catálogo limpio: {df_full.shape[0]} canciones")

# Para que todo corra rápido en clase trabajamos con una muestra,
# y reservamos 500 canciones "que llegarán mañana al catálogo" para la Fase 6.
muestra = df_full.sample(n=5500, random_state=RANDOM_STATE)
df_work = muestra.iloc[:5000].reset_index(drop=True)   # tu dataset de trabajo
df_new  = muestra.iloc[5000:].reset_index(drop=True)   # canciones nuevas: ¡ni las mires todavía!

print(f"df_work: {df_work.shape}   df_new: {df_new.shape}")


### Diccionario de datos

| Columna | Qué es |
|---|---|
| `track_id`, `track_name`, `artists`, `album_name` | Identificadores y texto |
| `popularity` | Popularidad 0-100 (calculada por Spotify) |
| `duration_ms` | Duración en milisegundos |
| `explicit` | Contenido explícito (True/False) |
| `danceability`, `energy`, `valence` | Cómo de bailable, enérgica y "alegre" suena (0-1) |
| `acousticness`, `instrumentalness`, `speechiness`, `liveness` | Prob. de ser acústica, instrumental, hablada, en directo (0-1) |
| `loudness` | Volumen medio en dB (negativo: cuanto más cerca de 0, más fuerte suena) |
| `tempo` | Velocidad en BPM |
| `key`, `mode`, `time_signature` | Tonalidad (0-11), modo mayor/menor, compás |
| `track_genre` | Género asignado a la pista |


---
# Fase 1 · EDA e ingeniería de variables (25 min)

**Objetivo de la fase:** decidir **qué variables entran** al clustering y dejarlas listas en una matriz escalada `X_scaled`.

Recuerda lo visto en Ingeniería de Variables I y II:
- Identifica el **tipo** de cada variable antes de tocarla (¿numérica de verdad? ¿pseudo-numérica? ¿texto?).
- Las **colas largas** dominan la distancia -> `log1p`.
- La **escala** manda en cualquier modelo de distancias -> escalar SIEMPRE antes de clustering.

Y dos preguntas trampa que responderás en la mini reflexión:
- ¿Debe entrar `popularity`? Pista: ¿describe cómo **suena** la canción o cómo le **fue**?
- ¿Debe entrar `track_genre`? Pista: ¿qué pasa si agrupas usando (casi) un target?


In [ ]:
df_work.info()
df_work.describe().T[["mean", "std", "min", "max"]].round(2)


In [ ]:
cols_numericas = ["popularity", "duration_ms", "danceability", "energy", "loudness",
                  "speechiness", "acousticness", "instrumentalness", "liveness",
                  "valence", "tempo"]

df_work[cols_numericas].hist(bins=40, figsize=(14, 8), layout=(3, 4))
plt.suptitle("Distribuciones: busca escalas distintas y colas largas")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 7))
sns.heatmap(df_work[cols_numericas].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlaciones (fíjate en energy-loudness y energy-acousticness)")
plt.show()


In [ ]:
# Decisiones de ingeniería de variables (cada una es discutible; lo importante es justificarla):
# - Fuera identificadores y texto: track_id, track_name, artists, album_name
# - Fuera popularity: no describe cómo SUENA la canción sino cómo le fue -> la guardamos para el profiling
# - Fuera track_genre: es (casi) un target -> lo reservamos como validación externa en la Fase 5
# - Fuera time_signature: casi todo es 4/4, apenas aporta varianza (selección de variables, IV II)
# - Fuera mode y explicit: binarias; podrían entrar, pero hoy centramos el clustering en el sonido continuo
# - duration_ms tiene cola larga -> pasamos a minutos y aplicamos log1p (IV I)

df_work["duration_min_log"] = np.log1p(df_work["duration_ms"] / 60000)

features = ["danceability", "energy", "loudness", "speechiness", "acousticness",
            "instrumentalness", "liveness", "valence", "tempo", "duration_min_log"]

print(f"{len(features)} features seleccionadas:", features)

# [EXTRA] key es circular: la tonalidad 11 y la 0 son vecinas, igual que las 23h y las 0h.
# Así se codificaría (hoy la dejamos fuera para no complicar el perfil de la Fase 5):
df_work["key_sin"] = np.sin(2 * np.pi * df_work["key"] / 12)
df_work["key_cos"] = np.cos(2 * np.pi * df_work["key"] / 12)


In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(df_work[features]), columns=features)

X_scaled.describe().T[["mean", "std", "min", "max"]].round(2)


### ✍️ Mini reflexión (2 min)

- ¿Por qué has dejado fuera `popularity` y `track_genre`?
- ¿Qué le pasaría a la distancia entre canciones si NO escalas (`tempo` va de 0 a 240, `danceability` de 0 a 1)?

**Respuesta:**

- `popularity` no describe el sonido: mezcla marketing, época y azar. Si entra, el clustering agrupa por éxito y no por tipo de canción (y el encargo pide tipos de canción). `track_genre` es una etiqueta humana: agrupar usándola sería hacer trampas al solitario; mejor reservarla para validar al final si los grupos tienen sentido.
- Sin escalar, `tempo` (0-240) y `loudness` (-50 a 4) dominarían la distancia euclídea: dos canciones idénticas en todo menos el tempo quedarían lejísimos, y `danceability` (0-1) no pintaría nada. Escalar pone a todas las variables a hablar con el mismo volumen.


In [ ]:
# --- CHECKPOINT 1 · Ejecuta esta celda tal cual (no la edites) ---
checks = [("Falta la lista `features` con las columnas elegidas (mínimo 5)",
           "features" in globals() and isinstance(features, list) and len(features) >= 5)]
if "scaler" not in globals():
    checks.append(("Falta `scaler` (el StandardScaler ajustado; lo necesitarás en la Fase 6)", False))
if "X_scaled" not in globals():
    checks.append(("Falta `X_scaled` (la matriz escalada lista para el modelo)", False))
else:
    _X = np.asarray(X_scaled)
    checks += [
        ("X_scaled tiene NaN: revisa nulos o transformaciones", not np.isnan(_X).any()),
        ("X_scaled debe tener una fila por canción de df_work", _X.shape[0] == len(df_work)),
        ("Las medias no están cerca de 0: ¿has escalado con StandardScaler?",
         bool(np.abs(_X.mean(axis=0)).max() < 0.1)),
        ("Las desviaciones no están cerca de 1: ¿has escalado?",
         bool(0.5 < _X.std(axis=0).mean() < 1.5)),
    ]
checkpoint(1, checks)


---
# Fase 2 · PCA: mira tus datos (15 min)

Tienes 10 dimensiones y los ojos solo soportan 2. **PCA** comprime las features en componentes que conservan la máxima varianza (IV II). Aquí lo usamos para dos cosas:

1. Saber **cuánta estructura** hay: ¿cuántas componentes hacen falta para el 90% de la varianza?
2. **Ver** el catálogo en 2D antes de agrupar (y en la Fase 3, pintar los clusters encima).

Ojo: escala SIEMPRE antes de PCA... que ya hiciste en la Fase 1. 😉


In [ ]:
pca = PCA(random_state=RANDOM_STATE).fit(X_scaled)
var = pca.explained_variance_ratio_
var_acum = np.cumsum(var)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1, len(var) + 1), var)
ax[0].set(title="Scree plot: varianza por componente", xlabel="Componente", ylabel="Varianza explicada")
ax[1].plot(range(1, len(var) + 1), var_acum, marker="o")
ax[1].axhline(0.9, ls="--", c="gray")
ax[1].set(title="Varianza explicada acumulada", xlabel="Nº de componentes")
plt.show()

n_90 = int(np.argmax(var_acum >= 0.9)) + 1
print(f"Con {n_90} componentes conservamos el 90% de la varianza (partíamos de {X_scaled.shape[1]})")


In [ ]:
X_pca2 = pca.transform(X_scaled)[:, :2]

plt.figure(figsize=(8, 6))
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], s=5, alpha=0.4)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var.)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var.)")
plt.title("El catálogo proyectado en 2D")
plt.show()

# [EXTRA] ¿Qué mezcla cada componente? Los pesos (loadings) ayudan a leer el mapa:
# PC1 enfrenta energy/loudness contra acousticness -> "intensidad del sonido"
pd.DataFrame(pca.components_[:2].T, index=features, columns=["PC1", "PC2"]).round(2)


### ✍️ Mini reflexión (2 min)

- ¿Cuánta varianza conservas en 2D? ¿Es este mapa una foto fiel o un resumen con pérdida?
- ¿Se ven grupos separados a simple vista o una nube continua? ¿Demuestra eso que no hay clusters?

**Respuesta:**

- PC1+PC2 conservan en torno al 44% de la varianza: el mapa es un resumen con pérdida. Dos puntos juntos aquí pueden estar lejos en las 10 dimensiones reales (y al revés). Sirve para orientarse, no para sentenciar.
- Se ve una nube más bien continua con zonas densas. Eso NO demuestra que no haya clusters: en 2D solo vemos parte de la señal. La música es un continuo; los algoritmos van a "trocear" ese continuo, y aún así los trozos pueden ser útiles para producto.


In [ ]:
# --- CHECKPOINT 2 · Ejecuta esta celda tal cual ---
checks = []
if "pca" not in globals() or not hasattr(pca, "explained_variance_ratio_"):
    checks.append(("Falta un PCA ajustado en la variable `pca`", False))
if "X_pca2" not in globals():
    checks.append(("Falta `X_pca2` con la proyección en 2D", False))
else:
    _P = np.asarray(X_pca2)
    checks += [
        ("X_pca2 debe tener exactamente 2 columnas", _P.ndim == 2 and _P.shape[1] == 2),
        ("X_pca2 debe tener una fila por canción", _P.shape[0] == len(df_work)),
    ]
checkpoint(2, checks)


---
# Fase 3 · K-Means: tu primer modelo (25 min)

Toca el caballo de batalla del clustering. Recuerda el bucle: **asignar -> recalcular centros -> repetir**, con `k-means++` como arranque por defecto. La decisión importante no es entrenar (son 2 líneas), es **elegir K**:

- **Codo**: dónde la inercia deja de bajar con ganas.
- **Silhouette**: cómo de bien asignado está cada punto (de -1 a 1, más alto mejor).
- **Negocio**: ¿cuántos segmentos puede accionar producto de verdad?

Ninguno de los tres manda solo. Vas a calcular los dos primeros y a decidir con los tres.


In [ ]:
inercias, siluetas = [], []
rango_k = range(2, 9)

for k in rango_k:
    km_k = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inercias.append(km_k.inertia_)
    siluetas.append(silhouette_score(X_scaled, km_k.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(rango_k), inercias, marker="o")
ax[0].set(title="Método del codo", xlabel="K", ylabel="Inercia")
ax[1].plot(list(rango_k), siluetas, marker="o")
ax[1].set(title="Silhouette medio", xlabel="K")
plt.show()

pd.DataFrame({"K": list(rango_k), "inercia": np.round(inercias), "silhouette": np.round(siluetas, 3)})


In [ ]:
# El silhouette máximo cae en K=2... pero 2 segmentos ("tranquilas" vs "enérgicas") no dan
# para un producto de playlists. Entre K=4 y K=6 el silhouette es casi plano (~0.15), así que
# el dato deja de decidir y decide el negocio: K=5 da segmentos de tamaño razonable y
# perfiles claros. Esto es "defiende tu K": números + interpretabilidad + acción.
k_elegido = 5

km = KMeans(n_clusters=k_elegido, init="k-means++", n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
labels_km = km.labels_
sil_km = silhouette_score(X_scaled, labels_km)

print(f"K = {k_elegido} | silhouette = {sil_km:.3f}")
print("Canciones por cluster:", np.bincount(labels_km))

plt.figure(figsize=(8, 6))
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=labels_km, cmap="tab10", s=5, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-Means con K={k_elegido} sobre el mapa PCA")
plt.show()


### ✍️ Mini reflexión (2 min)

- ¿El codo y el silhouette apuntan al mismo K? Si no, ¿con qué criterio desempatas?
- Si el silhouette máximo sale en K=2, ¿por qué podría aun así NO ser la mejor entrega para producto?

**Respuesta:**

- Aquí no coinciden: el silhouette máximo está en K=2 (≈0.25) y el codo es suave, sin quiebro claro. El desempate no es estadístico sino de negocio: ¿cuántos segmentos distintos y accionables necesita el caso de uso? Entre K=4 y K=6 el silhouette apenas cambia (≈0.15), señal de que cualquier K de ese rango es defendible.
- K=2 separa "calmado" vs "enérgico": correcto pero pobre; producto no puede montar una estrategia de playlists con 2 cubos. Un buen entregable sacrifica algo de métrica interna a cambio de segmentos con significado. Eso sí, se documenta: se elige K=5 con silhouette 0.15 frente al 0.25 de K=2, y se explica el porqué.


In [ ]:
# --- CHECKPOINT 3 · Ejecuta esta celda tal cual ---
checks = [("Falta `k_elegido` (un entero entre 2 y 10)",
           "k_elegido" in globals() and isinstance(k_elegido, (int, np.integer)) and 2 <= k_elegido <= 10)]
if "labels_km" not in globals():
    checks.append(("Faltan las etiquetas `labels_km` del K-Means final", False))
else:
    _L = np.asarray(labels_km)
    checks.append(("labels_km debe tener una etiqueta por canción", len(_L) == len(df_work)))
    if "k_elegido" in globals():
        checks.append(("El nº de clusters de labels_km no coincide con k_elegido",
                       len(set(_L.tolist())) == k_elegido))
checks.append(("Falta `sil_km` (el silhouette del modelo final)",
               "sil_km" in globals() and -1 <= float(sil_km) <= 1))
checkpoint(3, checks)


---
## ☕ Descanso (5 min)

Has cerrado la mitad del workshop: datos -> features -> PCA -> primer modelo. Estira las piernas. Al volver: retadores, perfiles y producción.


---
# Fase 4 · El retador: elige tu segundo modelo (25 min)

K-Means asume grupos compactos y "redondos". Toca ponerlo a prueba con un **retador**. Elige **UNA** opción [MÍNIMO] (o varias si vas sobrado [EXTRA]):

| Opción | Modelo | Para qué brilla | Detalle práctico |
|---|---|---|---|
| A | **Jerárquico (ward)** | Ver la estructura con el dendrograma y cortar donde tenga sentido | Es O(n²): usa una submuestra de ~2.000 canciones |
| B | **DBSCAN** | Formas irregulares y detección de ruido (canciones "inclasificables") | En 10D las distancias se difuminan: pruébalo sobre las 3 primeras componentes PCA |
| C | **K-Medoids** | El centro de cada grupo es una **canción real** que puedes escuchar | `pip install kmedoids` + matriz de distancias en una submuestra |

Guarda al final:
- `nombre_reto` = `"jerarquico"`, `"dbscan"` o `"kmedoids"`
- `labels_reto` = etiquetas de tu retador

⚠️ Juego limpio al comparar: el silhouette solo es comparable **sobre los mismos datos y el mismo espacio**. Si usaste submuestra o PCA, dilo en tu conclusión. Y en DBSCAN, calcula el silhouette sin los puntos de ruido (etiqueta -1) e indica el % de ruido.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Submuestra: el jerárquico es O(n²) y con 5.000 canciones ya sufre
idx_sub = np.random.default_rng(RANDOM_STATE).choice(len(X_scaled), size=2000, replace=False)
X_sub = X_scaled.iloc[idx_sub].to_numpy()

Z = linkage(X_sub, method="ward")

plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.title("Dendrograma (ward, truncado a 30 ramas)")
plt.ylabel("Distancia de fusión")
plt.show()

labels_jer = fcluster(Z, t=k_elegido, criterion="maxclust")
sil_jer = silhouette_score(X_sub, labels_jer)
print(f"Jerárquico con K={k_elegido} (submuestra): silhouette {sil_jer:.3f}")
print("Tamaños:", np.bincount(labels_jer)[1:])


In [ ]:
# En el espacio original (10D escaladas) DBSCAN lo pasa mal: con eps pequeño todo es ruido
# y con eps grande todo es un único cluster. Es la maldición de la dimensionalidad (IV II).
# Lo aplicamos sobre las 3 primeras componentes del PCA (~57% de la varianza):
X_pca3 = pca.transform(X_scaled)[:, :3]

for eps in [0.3, 0.5, 0.7, 0.9]:
    db_e = DBSCAN(eps=eps, min_samples=15).fit(X_pca3)
    n_c = len(set(db_e.labels_.tolist()) - {-1})
    print(f"eps={eps}: {n_c} clusters, {(db_e.labels_ == -1).mean():.0%} de ruido")

# eps=0.5 da un equilibrio razonable: estructura + un ~14% de canciones "inclasificables"
db = DBSCAN(eps=0.5, min_samples=15).fit(X_pca3)
labels_db = db.labels_
mask = labels_db != -1
sil_db = silhouette_score(X_pca3[mask], labels_db[mask])
print(f"\nDBSCAN eps=0.5: {len(set(labels_db.tolist()) - {-1})} clusters, "
      f"{(~mask).mean():.0%} de ruido, silhouette sin ruido {sil_db:.3f} (¡en el espacio PCA 3D!)")


In [ ]:
!pip install -q kmedoids

from kmedoids import KMedoids
from scipy.spatial.distance import pdist, squareform

# Reutilizamos la submuestra del jerárquico (créala igual si no hiciste la opción A)
idx_sub = np.random.default_rng(RANDOM_STATE).choice(len(X_scaled), size=2000, replace=False)
X_sub = X_scaled.iloc[idx_sub].to_numpy()

D = squareform(pdist(X_sub))
kmed = KMedoids(n_clusters=k_elegido, metric="precomputed", random_state=RANDOM_STATE).fit(D)
labels_kmed = kmed.labels_.astype(int)
sil_kmed = silhouette_score(X_sub, labels_kmed)
print(f"K-Medoids con K={k_elegido} (submuestra): silhouette {sil_kmed:.3f}")

# La gracia de K-Medoids: el centro de cada grupo es una CANCIÓN REAL.
# Escúchalas y tendrás medio profiling hecho.
medoides = df_work.iloc[idx_sub[kmed.medoid_indices_]][["track_name", "artists", "track_genre"]].copy()
medoides.insert(0, "cluster", range(k_elegido))
medoides


In [ ]:
# Nos quedamos con el jerárquico como retador oficial (elige el tuyo)
nombre_reto = "jerarquico"
labels_reto = labels_jer

comparacion = pd.DataFrame({
    "modelo": ["K-Means (5.000 canciones)", "Jerárquico ward (submuestra 2.000)",
               "DBSCAN eps=0.5 (espacio PCA 3D)", "K-Medoids (submuestra 2.000)"],
    "grupos": [k_elegido, k_elegido, len(set(labels_db.tolist()) - {-1}), k_elegido],
    "silhouette": [round(sil_km, 3), round(sil_jer, 3), round(sil_db, 3), round(sil_kmed, 3)],
    "nota": ["referencia", "estructura parecida a K-Means",
             "sil no comparable (otro espacio); aporta el concepto de ruido",
             "similar a K-Means pero con centros interpretables"],
})
comparacion


### ✍️ Mini reflexión (2 min)

- ¿Tu retador ha encontrado una estructura distinta a la de K-Means o básicamente la misma?
- ¿En qué caso de negocio preferirías DBSCAN aunque su silhouette no sea comparable? ¿Y K-Medoids?

**Respuesta:**

- Con estos datos, jerárquico y K-Medoids encuentran una estructura muy parecida a la de K-Means (los grupos "de verdad" del sonido son bastante convexos). DBSCAN cuenta otra historia: no busca K grupos, separa zonas densas y marca ~14% de canciones como ruido.
- DBSCAN gana cuando el valor está en las anomalías (fraude, sensores) o los grupos tienen formas raras; aquí, ese 14% "inclasificable" sería oro para un equipo de contenido nicho. K-Medoids gana cuando necesitas explicar los grupos a negocio con un ejemplo real ("este segmento es, literalmente, esta canción") o cuando hay outliers que arrastran a los centroides.


In [ ]:
# --- CHECKPOINT 4 · Ejecuta esta celda tal cual ---
checks = [("Falta `nombre_reto` (texto: 'jerarquico', 'dbscan' o 'kmedoids')",
           "nombre_reto" in globals() and isinstance(nombre_reto, str))]
if "labels_reto" not in globals() or labels_reto is Ellipsis:
    checks.append(("Faltan las etiquetas `labels_reto` de tu modelo retador", False))
else:
    _L2 = np.asarray(labels_reto)
    _n_grupos = len(set(_L2.tolist()) - {-1})
    checks.append(("Tu retador debe encontrar al menos 2 clusters (sin contar el ruido -1)",
                   _n_grupos >= 2))
checkpoint(4, checks)


---
# Fase 5 · Profiling: de números a segmentos (20 min)

Un cluster sin nombre no sirve. Aplica el profiling en 4 pasos de la clase de Análisis Cluster:

1. **Perfil** de cada grupo: media por variable, sobre las variables ORIGINALES, no las escaladas (nadie entiende "energy = 0.7 desviaciones").
2. **Índice vs media global**: divide el perfil de cada grupo entre la media de todo el catálogo. Un 2.0 significa "el doble que la media".
3. **Tamaño** de cada grupo: ¿es un segmento o una anécdota?
4. **Nombre y acción**: el entregable de verdad.

Extra de hoy: guardamos `track_genre` y `popularity` fuera del modelo. Úsalas ahora como **validación externa**: si un cluster concentra sleep/ambient, vamos bien.

⚠️ Ojo con `loudness`: es negativa (dB), así que su índice se lee al revés (1.7 = más negativa = MÁS silenciosa). Con variables negativas, mira el perfil en crudo antes de interpretar el índice.


In [ ]:
df_work["cluster"] = labels_km

cols_perfil = features + ["popularity"]
perfil = df_work.groupby("cluster")[cols_perfil].mean()
perfil["n_canciones"] = df_work.groupby("cluster").size()
print(perfil.round(2).to_string())

indice = perfil[cols_perfil] / df_work[cols_perfil].mean()

plt.figure(figsize=(11, 4))
sns.heatmap(indice, annot=True, fmt=".2f", cmap="RdBu_r", center=1)
plt.title("Índice vs media global (1.0 = media del catálogo)")
plt.show()


In [ ]:
for c in sorted(df_work["cluster"].unique()):
    sub = df_work[df_work["cluster"] == c]
    top = ", ".join(f"{g} ({n})" for g, n in sub["track_genre"].value_counts().head(3).items())
    print(f"Cluster {c} · {len(sub)} canciones · popularidad media {sub['popularity'].mean():.0f}")
    print(f"   géneros top: {top}")


In [ ]:
# Los perfiles exactos dependen de tu muestra y tu K. Con K=5 y semilla 42 salen estos
# (léelos con tu heatmap y tus géneros top delante):
nombres_clusters = {
    0: "Voz en directo",        # speechiness x5.6, liveness x1.7 -> comedia y palabra hablada
    1: "Muro de sonido",        # energía alta, instrumental, valence bajo -> metal y techno
    2: "Calma instrumental",    # silenciosa, acústica, instrumental -> sleep, ambient, new-age
    3: "Acústicas de sofá",     # acústica x2, energía media, cantada -> tango, romance, cantopop
    4: "Fiesta alegre",         # bailable, valence x1.5 y lo más popular -> salsa, dance
}

df_work["segmento"] = df_work["cluster"].map(nombres_clusters)
df_work["segmento"].value_counts()


**Acción de producto por segmento** (ejemplo con los 5 de arriba):

- **Voz en directo**: separarlo del universo "música"; alimenta la sección de podcasts/comedia y evita que contamine las playlists musicales.
- **Muro de sonido**: playlists de intensidad ("Gym rage", "Deep focus techno") segmentadas por hora del día.
- **Calma instrumental**: playlists funcionales de dormir/estudiar; candidata a autoplay nocturno.
- **Acústicas de sofá**: playlist "Domingo tranquilo"; buen inventario para moods regionales (tango, cantopop).
- **Fiesta alegre**: el segmento más popular; playlists sociales y campañas de fin de semana.


In [ ]:
# --- CHECKPOINT 5 · Ejecuta esta celda tal cual ---
checks = []
if "perfil" not in globals() or not isinstance(perfil, pd.DataFrame):
    checks.append(("Falta la tabla `perfil` (DataFrame con el perfil medio por cluster)", False))
elif "k_elegido" in globals():
    checks.append(("`perfil` debe tener una fila por cluster", len(perfil) == k_elegido))
checks.append(("Falta `indice` (perfil dividido entre la media global; el bonus lo usa)",
               "indice" in globals()))
if "nombres_clusters" not in globals() or not isinstance(nombres_clusters, dict) or len(nombres_clusters) < 2:
    checks.append(("Falta `nombres_clusters` (diccionario {número de cluster: nombre})", False))
else:
    checks.append(("Los nombres no pueden ser 'Cluster 0', 'Cluster 1'... ponles nombre de verdad",
                   not any(str(v).strip().lower().startswith("cluster") for v in nombres_clusters.values())))
checkpoint(5, checks)


---
# Fase 6 · De cluster a producto: KNN (15 min)

El clustering está entregado... pero mañana entran canciones nuevas al catálogo y hay que asignarlas a un segmento **sin reentrenar nada**. Aquí conectamos con la clase de **KNN (eager vs lazy)**:

- Un clasificador *eager* aprende una función y se olvida del dataset.
- **KNN es *lazy***: memoriza el catálogo etiquetado y, cuando llega una canción nueva, mira sus K vecinos más cercanos y vota.

Es el cierre del círculo: el clustering (no supervisado) ha **creado las etiquetas** que ahora un clasificador (supervisado) aprende a asignar. Este patrón (clustering -> etiquetas -> clasificador) se usa muchísimo en proyectos reales.

Las 500 canciones "de mañana" te esperan en `df_new` desde la Fase 0.


In [ ]:
# Misma receta, mismo scaler. Hacer fit() aquí sería una fuga silenciosa: las canciones
# nuevas redefinirían la escala con la que se calculó todo lo anterior.
df_new["duration_min_log"] = np.log1p(df_new["duration_ms"] / 60000)
X_new_scaled = pd.DataFrame(scaler.transform(df_new[features]), columns=features)
print(X_new_scaled.shape)


In [ ]:
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_scaled, labels_km)
pred_nuevas = knn.predict(X_new_scaled)

print("Canciones nuevas por segmento:")
print(pd.Series(pred_nuevas).map(nombres_clusters).value_counts())

acuerdo = (pred_nuevas == km.predict(X_new_scaled)).mean()
print(f"\nAcuerdo KNN vs centroide más cercano: {acuerdo:.1%}")
# Un acuerdo alto (aquí ~94%) dice que ambos mecanismos son consistentes. KNN además
# seguiría funcionando con clusters de forma rara, donde el centroide engaña.


In [ ]:
# Escalamos TODO el catálogo (~90.000 canciones) con el scaler ya ajustado
df_cat = df_full.copy()
df_cat["duration_min_log"] = np.log1p(df_cat["duration_ms"] / 60000)
X_cat = scaler.transform(df_cat[features])

nn = NearestNeighbors(n_neighbors=11).fit(X_cat)

MI_CANCION = "Blinding Lights"   # cámbiala por la tuya

candidatas = df_cat[df_cat["track_name"].str.contains(MI_CANCION, case=False, na=False)]
# Si hay varias versiones, nos quedamos con la más popular
semilla = candidatas.sort_values("popularity", ascending=False).index[0]
print("Canción semilla:", df_cat.loc[semilla, "track_name"], "·", df_cat.loc[semilla, "artists"])

dist, vecinos = nn.kneighbors(X_cat[[semilla]])
playlist = df_cat.iloc[vecinos[0][1:]][["track_name", "artists", "track_genre", "popularity"]]
playlist


### ✍️ Mini reflexión (2 min)

- Tu recomendador solo usa features de audio. ¿Qué le falta frente al recomendador real de Spotify?

**Respuesta:**

Le falta el comportamiento: qué escuchan juntos los usuarios (filtrado colaborativo), saltos, likes, contexto (hora, dispositivo), letras y metadatos editoriales. Dos canciones pueden sonar parecidas y vivir en mundos distintos. El audio es una señal útil, sobre todo para canciones nuevas sin historial (cold start), pero el sistema real combina varias señales.


In [ ]:
# --- CHECKPOINT 6 · Ejecuta esta celda tal cual ---
checks = []
if "pred_nuevas" not in globals():
    checks.append(("Falta `pred_nuevas` (cluster asignado a cada canción nueva)", False))
else:
    _p = np.asarray(pred_nuevas)
    checks.append(("pred_nuevas debe tener una predicción por canción de df_new",
                   len(_p) == len(df_new)))
    if "labels_km" in globals():
        checks.append(("pred_nuevas contiene clusters que no existen en labels_km",
                       set(np.unique(_p).tolist()) <= set(np.unique(np.asarray(labels_km)).tolist())))
checkpoint(6, checks)


---
# 🤖 Bonus · Ponles nombre con un LLM (si sobra tiempo)

Los nombres de la Fase 5 los pusiste tú. Ahora deja que un LLM proponga los suyos **a partir de tu tabla de índices**: buen ejemplo de cómo combinar clustering + IA generativa (el modelo no ve las canciones, solo tu perfil agregado).

Consigue una API key gratuita en [OpenRouter](https://openrouter.ai/settings/keys). Si no tienes key, imprime el prompt y pégalo en cualquier chat de IA: el ejercicio es el mismo.


In [ ]:
# Construimos el prompt directamente desde tu tabla de índices (tu análisis alimenta al LLM)
if "indice" not in globals():
    print("Termina la Fase 5 primero: este bonus necesita tu tabla `indice`.")
else:
    tabla = indice.round(2).to_string()
    prompt = f"""Eres analista musical en Spotify. Esta tabla resume {k_elegido} clusters de canciones
como índice frente a la media del catálogo (1.0 = media, 2.0 = el doble, 0.5 = la mitad).
Ojo: loudness es negativa, así que un índice alto significa MÁS silenciosa.

{tabla}

Para cada cluster propón: nombre de playlist (máximo 4 palabras), descripción de una frase
y el momento del día para escucharla. Responde en una tabla markdown."""
    print(prompt)


In [ ]:
API_KEY = "TU-API-KEY"   # https://openrouter.ai/settings/keys

if API_KEY == "TU-API-KEY" or "prompt" not in globals():
    print("Sin API key: copia el prompt de arriba en cualquier chat de IA y compara sus nombres con los tuyos.")
else:
    from openai import OpenAI
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
    respuesta = client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
    )
    print(respuesta.choices[0].message.content)


---
# 🏁 Cierre del workshop


In [ ]:
# --- Tu progreso final ---
total = sum(bool(v) for v in CHECKPOINTS.values())
print("=" * 46)
print(f"   WORKSHOP: {total} de 6 checkpoints superados")
print("=" * 46)
for i in range(1, 7):
    estado = "✅" if CHECKPOINTS.get(i) else "⬜"
    print(f" {estado}  Checkpoint {i}")
if total == 6:
    print("\n🏆 End-to-end completo: EDA -> features -> PCA -> clustering -> profiling -> producción.")
else:
    print("\nPuedes volver a cualquier fase y cerrar los checkpoints que falten.")


### Preguntas finales (para discutir en grupo)

1. De las 8 fases del plan de trabajo, ¿cuál te ha llevado más tiempo? ¿Cuál llevaría más tiempo en un proyecto real?
2. ¿Qué harías distinto si en vez de canciones fueran clientes de un banco? ¿Qué cambia y qué no cambia del plan?
3. El silhouette "prefería" K=2 y tú probablemente entregaste otro K. ¿Cómo se lo explicas a un perfil técnico que te acusa de ignorar la métrica?
4. ¿Qué habría que monitorizar si este sistema (clustering + KNN) se pone en producción de verdad?

**Respuestas orientativas:**

1. En clase, el modelado. En un proyecto real: el EDA + la ingeniería de variables (y definir el problema, que hoy venía regalado en el encargo).
2. Cambian los datos (RFM, transacciones, canales) y la acción (campañas en vez de playlists); NO cambia el plan: escalar, elegir K con métrica + negocio, perfilar, activar. Ese es el punto de tener un plan de trabajo.
3. "El silhouette mide geometría, no valor: con K=2 la partición es más limpia pero inaccionable. Entre K=4 y K=6 la métrica es plana (~0.15), así que la decisión pasa a criterios de producto, y queda documentada."
4. Drift de las features (¿cambia la distribución del catálogo?), tamaño y estabilidad de los segmentos en el tiempo, % de canciones "lejos" de todo segmento (candidatas a ruido o a un segmento nuevo) y consistencia KNN vs centroide como alarma barata.

### Takeaways

- El plan de trabajo no supervisado es siempre el mismo: **datos -> features escaladas -> (PCA) -> modelo + K justificado -> profiling -> activación**.
- Las métricas (codo, silhouette) **orientan pero no deciden**: el K final se defiende con números + interpretabilidad + acción.
- Cada algoritmo asume una forma de grupo: K-Means (compactos), jerárquico (árbol interpretable), DBSCAN (densidad + ruido), K-Medoids (centros reales).
- El profiling es el entregable: un cluster sin nombre y sin acción no genera valor.
- El ciclo se cierra en producción: el clustering crea etiquetas, KNN las asigna a datos nuevos con la MISMA receta de features y sin fugas.
